# Notebook Python confusion generation

In [1]:
import pandas as pd
import json

In [2]:
df = pd.read_csv("Metagenomic_gold_standard_consensus.tsv", sep="\t", encoding="utf-8")

print("Columns detected in file:")
for col in df.columns:
    print(f"[{col}]")

Columns detected in file:
[Source]
[Tool name]
[Topic]
[Operation]
[Input data]
[Output data]
[Input format]
[Output format]


In [ ]:
def process_annotation_field(field_value):
    """
    Process annotation fields by:
    - converting to lowercase
    - removing excess whitespace
    - splitting on commas
    - returning a clean list of annotations
    """
    if pd.isna(field_value) or field_value == "" or field_value is None:
        return []

    # Convert to lowercase
    text = str(field_value).lower()

    # Normalize excessive whitespace
    text = " ".join(text.split())

    # Split by commas and strip whitespace
    annotations = [item.strip() for item in text.split(",")]

    # Remove empty strings
    annotations = [item for item in annotations if item]

    return annotations


def load_tsv_and_create_json(
    tsv_file="Metagenomic_gold_standard_consensus.tsv",
    json_file="metagenomic_tools_annotations.json",
):
    """
    Load TSV and create JSON including all sources (consensus_expert_LLM, proposition DeepSeek-V3.1, consensus_expert)
    """
    df = pd.read_csv(tsv_file, sep="\t", encoding="utf-8")
    df.columns = df.columns.str.strip()  # normalize column names

    all_tools = {}
    counter = 1

    for _, row in df.iterrows():
        tool_key = f"tools#{counter}"
        source = row.get("Source", "").strip()

        # Parse all annotation columns robustly
        tool_data = {
            "name": row.get("Tool name", "").strip(),
            "discipline": "Metagenomics",
            "source": source,
            "topic": process_annotation_field(row.get("Topic", "")),
            "operation": process_annotation_field(row.get("Operation", "")),
            "input data": process_annotation_field(row.get("Input data", "")),
            "output data": process_annotation_field(row.get("Output data", "")),
            "input format": process_annotation_field(row.get("Input format", "")),
            "output format": process_annotation_field(row.get("Output format", "")),
        }

        all_tools[tool_key] = tool_data
        counter += 1

    # Save JSON
    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(all_tools, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(all_tools)} tools to {json_file}")


In [5]:
def load_json(json_file="metagenomic_tools_annotations.json"):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading JSON: {e}")
        return None


In [6]:
def calculate_metrics_per_tool_annotation(json_data):
    """
    For each tool and each annotation type, calculate TP, FP, FN, TN, precision, recall, F1
    using lowercase cleaned annotations from process_annotation_field().
    """

    annotation_fields = [
        "topic",
        "operation",
        "input data",
        "output data",
        "input format",
        "output format",
    ]

    results = []

    # collect all unique tool names (already lowercase)
    tool_names = set(entry["name"] for entry in json_data.values())

    # expected provenances (lowercase, normalized)
    expected_prov = {
        "consensus expert llm",
        "proposition deepseek-v3.1",
        "consensus expert",
    }

    for tool in tool_names:
        # initialize empty sets per provenance
        prov_sets = {
            "consensus expert llm": {field: set() for field in annotation_fields},
            "proposition deepseek-v3.1": {field: set() for field in annotation_fields},
            "consensus expert": {field: set() for field in annotation_fields},
        }

        # fill annotation sets
        for entry in json_data.values():
            if entry["name"] != tool:
                continue

            prov = entry["source"].strip().lower()

            if prov not in prov_sets:
                continue

            for field in annotation_fields:
                prov_sets[prov][field].update(entry.get(field, []))

        # compute confusion metrics for each annotation type
        for field in annotation_fields:
            llm_set = prov_sets["consensus expert llm"][field]
            deepseek_set = prov_sets["proposition deepseek-v3.1"][field]
            expert_set = prov_sets["consensus expert"][field]

            # missing by deepseek
            missing_in_deepseek = llm_set - deepseek_set

            # FN = elements in LLM not predicted by DeepSeek but present in Expert
            FN_set = {x for x in missing_in_deepseek if x in expert_set}

            # mixed contributions = predicted by both even if not matched DeepSeek
            mixed_tp = {x for x in missing_in_deepseek if x not in expert_set}

            # TP: direct intersection + mixed tp
            TP_set = (llm_set & deepseek_set) | mixed_tp

            # FP: predicted by deepseek but not in LLM
            FP_set = deepseek_set - llm_set

            TN_set = set()  # always empty

            TP = len(TP_set)
            FP = len(FP_set)
            FN = len(FN_set)
            TN = 0

            precision = TP / (TP + FP) if (TP + FP) else 0
            recall = TP / (TP + FN) if (TP + FN) else 0
            f1_score = (
                2 * precision * recall / (precision + recall)
                if (precision + recall)
                else 0
            )

            results.append({
                "tool_name": tool,
                "annotation_type": field,
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "TN": TN,
                "precision": precision,
                "recall": recall,
                "f1_score": f1_score,
                "TP_annotations": sorted(TP_set),
                "FP_annotations": sorted(FP_set),
                "FN_annotations": sorted(FN_set),
            })

    return results


In [7]:
def save_metrics_to_tsv(metrics, tsv_file="confusion_metrics_per_tool_annotation.tsv"):
    df = pd.DataFrame(metrics)
    df.to_csv(tsv_file, sep="\t", index=False)
    print(f"Saved confusion metrics per tool and annotation type to {tsv_file}")


In [8]:
def main():
    # 1️⃣ Regenerate JSON including consensus_expert
    load_tsv_and_create_json(
        tsv_file="Metagenomic_gold_standard_consensus.tsv",
        json_file="metagenomic_tools_annotations.json",
    )

    # 2️⃣ Load JSON
    data = load_json()
    if not data:
        return

    # 3️⃣ Calculate metrics per tool and annotation type
    metrics = calculate_metrics_per_tool_annotation(data)

    # 4️⃣ Save to TSV
    save_metrics_to_tsv(metrics)


if __name__ == "__main__":
    main()


Saved 28 tools to metagenomic_tools_annotations.json


KeyError: 'source'